# NN · Название эксперимента

**Как пользоваться.** Скопируй этот ноутбук в `notebooks/NN_короткое_имя.ipynb`.
Меняется только ячейка **Конфиг** — остальное запускается как есть.
В конце впиши вывод в `exp.report(...)` — он попадёт в `docs/JOURNAL.md`.

> **Протокол** (`docs/HYPOTHESES.md`, `AGENTS.md`): вектор строится только на train,
> подбор — только на val, test открывается **один раз** на замороженный метод.
> Пайплайн это проверяет сам — если ячейка упала с `TestLockError` или `StageOrderError`,
> это защита сработала, а не баг.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

## 1. Что проверяем и зачем

- **Гипотеза:** H? — …
- **Что меняем относительно прошлых прогонов:** …
- **Что будет считаться успехом / провалом (заранее):** …

In [ ]:
cfg = ExperimentConfig.make(
    name="my-experiment",                 # коротко, латиницей: попадёт в run_id
    hypothesis="H3",                      # H1 | H2 | H3 | H4 | explore
    notebook="notebooks/NN_name.ipynb",   # путь к этому ноутбуку — для журнала
    notes="Зачем этот прогон, одной фразой.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",  # "mock" — без модели, за секунды
        # "data.n_tasks": 200,
        # "decision.mode": "generate",     # scored (быстро, по умолчанию) | generate
        # "steer.alphas": [0.5, 1, 2, 4],
        # "steer.windows": ["post_message"],
        # "eval.methods": ["baseline", "strong_prompt", "restate", "explicit_state",
        #                  "steering", "steering_gated", "tact", "lora"],
        # "episode.enabled": True,         # многошаговые эпизоды (episodes.py)
        # "episode.on_val": False,         # эпизоды только на test — быстрее
    },
)
for k, (old, new) in cfg.diff_from_default().items():
    print(f"{k:28} {old!r:>20}  ->  {new!r}")

In [ ]:
exp = Experiment(cfg)
exp.data()
exp.estimate()          # сколько займёт на этой машине — смотреть ДО запуска

## 2. База (материал для H2)

In [ ]:
exp.baseline()
exp.results_table("val")

## 3. Различает ли модель ответы — probe (H1)

In [ ]:
exp.probe()
viz.probe_heatmap(exp.probe_table());

## 4. Вектор (train) и подбор на validation (H3)

In [ ]:
exp.vectors()
exp.sweep()
viz.dose_response(exp.sweep_table(), cfg.steer.max_specificity_drop);
exp.state["selection"]

## 4b. Обучаемые конкуренты и сравнение всех методов на val

Гейт, TACT и LoRA учатся на train и выбираются на val тем же правилом, что и steering. Таблица ниже — картина **до** test; для выбранных методов val оптимистичен (на нём же подбор).

In [ ]:
exp.fit()
exp.results_table("val")

## 5. Заморозка и test — один раз

Перед этой ячейкой: результаты val просмотрены, конфиг больше не меняется.
Повторный запуск на тот же замороженный метод будет заблокирован.

In [ ]:
exp.freeze()
exp.test()
viz.compare_methods(exp.results_table("test"));
exp.results_table("test")

In [ ]:
for h, v in exp.verdicts().items():
    print(f"{h}: {v.label:16} {v.reason}")

## 6. Вывод → журнал

In [ ]:
display(Markdown(exp.report(
    conclusion="""Что показал прогон и насколько этому верить.""",
    next_step="""Что делаем дальше.""",
)))